In [ ]:
# what is Langgraph

# it is collection of Nodes + Agents + State

In [1]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [5]:
# Scenario: Customer Support Chatbot (Question-Based)
# Imagine a company has deployed a chatbot that answers customer
#  questions by calling the Groq API. The workflow is modeled as a
#  graph of states, where each customer query flows through nodes until
#   a response is delivered.

# 1. State Definition
# The chatbot maintains a notebook-like state:
# - question → The customer’s query.
# - answer → The response generated by Groq.
# - history → A log of all past questions.


from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    groq_api_key = userdata.get('API_KEY')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    # Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            # IMPORTANT: The list of supported models by Groq API changes frequently.
            # Please refer to the official Groq documentation (https://console.groq.com/docs/models)
            # to find a currently active and supported model name and replace 'YOUR_GROQ_MODEL_NAME_HERE' below.
            # Examples of often available models include 'llama3-8b-8192' or 'llama3-70b-8192',
            # but these can also become decommissioned.
            "model": "llama-3.1-8b-instant",   # Changed to a currently active model
            "messages": [{"role": "user", "content": q}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    # Extract answer from Groq response
    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    # Initial state
    state = {"question": "What are your store hours?", "answer": "", "history": []}

    # Run the graph
    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])

Bot says: I'm a large language model, I don't have a physical store. I exist solely as a digital entity, so I don't have store hours. I'm available 24/7 to provide information and assist with tasks.


In [8]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each learner:
# - topic → The subject the learner is studying (e.g., "Photosynthesis").
# - flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
# - progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

# 2. Workflow (Graph of States)
# Each learner interaction flows through nodes until a flashcard is delivered:
# - Input Node
# - Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
# - State updates: topic = "cell biology"
# - Generation Node (Groq API)
# - Groq generates a flashcard:
# - flashcard.question = "What organelle is known as the powerhouse of the cell?"
# - flashcard.answer = "Mitochondria"
# - Response Node
# - Assistant presents the flashcard question to the learner.
# - Evaluation Node
# - Learner responds with their answer.
# - Assistant checks correctness and updates progress.
# - History Node
# - Logs the flashcard attempt:
# - progress = [{question, learner_answer, correct/incorrect}]

from langgraph.graph import StateGraph, END
from typing import TypedDict, Dict, List
import requests
from google.colab import userdata


# 1. State
class StudyState(TypedDict):
    topic: str
    flashcard: Dict
    learner_answer: str
    progress: List


# 2. Generate flashcard
def generate_flashcard(state: StudyState):

    topic = state["topic"]
    groq_api_key = userdata.get("API_KEY")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": f"Create one flashcard about {topic}. Format: Question: ... Answer: ..."
                }
            ],
        },
    )

    text = response.json()["choices"][0]["message"]["content"]

    parts = text.split("Answer:")

    question = parts[0].replace("Question:", "").strip()
    answer = parts[1].strip()

    return {
        "flashcard": {
            "question": question,
            "answer": answer,
        }
    }


# 3. Show flashcard (NO input here)
def show_flashcard(state: StudyState):

    print("Question:", state["flashcard"]["question"])

    return state


# 4. Evaluate
def evaluate_answer(state: StudyState):

    correct = state["flashcard"]["answer"].lower()
    user = state["learner_answer"].lower()

    is_correct = correct in user

    print("Correct answer:", correct)
    print("Your answer:", user)
    print("Result:", is_correct)

    return {
        "progress": state["progress"] + [
            {
                "question": state["flashcard"]["question"],
                "learner_answer": user,
                "correct": is_correct,
            }
        ]
    }


# Graph
graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("show", show_flashcard)
graph.add_node("evaluate", evaluate_answer)

graph.set_entry_point("generate")

graph.add_edge("generate", "show")
graph.add_edge("show", "evaluate")
graph.add_edge("evaluate", END)


# Run
state = {
    "topic": "Photosynthesis",
    "flashcard": {},
    "learner_answer": "mitochondria",
    "progress": [],
}

app = graph.compile()

result = app.invoke(state)

print(result["progress"])

Question: Here's a flashcard about Photosynthesis:

  What is the process by which plants, algae, and some bacteria produce their food using sunlight?
Correct answer: photosynthesis
Your answer: mitochondria
Result: False
[{'question': "Here's a flashcard about Photosynthesis:\n\n  What is the process by which plants, algae, and some bacteria produce their food using sunlight?", 'learner_answer': 'mitochondria', 'correct': False}]


In [7]:
# Scenario: AI-Powered Project Tracker (Task-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each project:
# - task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
# - status → The current state of the task (e.g., "in progress", "completed", "blocked").
# - log → A history of all updates, including who made them and when.

# 2. Workflow (Graph of States)
# Each project update flows through nodes until the task status is refreshed:
# - Input Node
# - Team member submits an update (e.g., "Report draft completed").
# - State updates: task = "Q1 financial report"
# - Processing Node (Groq API)
# - Groq interprets the update and assigns a status:
# - status = "completed"
# - Response Node
# - Assistant confirms the update back to the team:
# - "Task Q1 financial report marked as completed."
# - History Node
# - Logs the update:
# - log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
from google.colab import userdata
from datetime import datetime


# 1. Define State
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    log: List


# 2. Processing Node (Groq API)
def process_update(state: ProjectState):

    task = state["task"]
    update = state["update"]

    groq_api_key = userdata.get("API_KEY")

    prompt = f"""
    Task: {task}
    Update: {update}

    Decide task status from:
    completed / in progress / blocked
    Return only one word.
    """

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {"role": "user", "content": prompt}
            ],
        },
    )

    status = response.json()["choices"][0]["message"]["content"].strip().lower()

    return {
        "status": status
    }


# 3. Response Node
def confirm_update(state: ProjectState):

    msg = f"Task {state['task']} marked as {state['status']}"

    print(msg)

    return state


# 4. History Node
def log_update(state: ProjectState):

    new_log = {
        "task": state["task"],
        "update": state["update"],
        "status": state["status"],
        "time": str(datetime.now())
    }

    return {
        "log": state["log"] + [new_log]
    }


# 5. Build Graph
graph = StateGraph(ProjectState)

graph.add_node("process", process_update)
graph.add_node("confirm", confirm_update)
graph.add_node("log", log_update)

graph.set_entry_point("process")

graph.add_edge("process", "confirm")
graph.add_edge("confirm", "log")
graph.add_edge("log", END)


# 6. Example Run
state = {
    "task": "Q1 financial report",
    "update": "Report draft completed",
    "status": "",
    "log": [],
}

app = graph.compile()

result = app.invoke(state)

print("\nLog history:")
print(result["log"])

Task Q1 financial report marked as completed

Log history:
[{'task': 'Q1 financial report', 'update': 'Report draft completed', 'status': 'completed', 'time': '2026-03-20 06:21:31.746738'}]


In [9]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


In [10]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"): # Changed model to a known working one
    groq_api_key = userdata.get('API_KEY')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def search_web(state: ResearchState):
    print(f"🔍 Searching: {state['topic']}")
    # Call Groq to generate snippets
    new_results = [
        groq_call(f"Give me a short article snippet about {state['topic']}"),
        groq_call(f"Give me another snippet about {state['topic']}")
    ]
    results = state["search_results"] + new_results
    return {
        "search_results": results,
        "steps_done": state["steps_done"] + 1
    }

def analyze_results(state: ResearchState):
    print(f"📊 Analyzing {len(state['search_results'])} results")
    joined_results = "\n".join(state["search_results"])
    analysis = groq_call(f"Analyze these sources and extract key insights:\n{joined_results}")
    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

def summarize(state: ResearchState):
    print("✍️ Generating summary...")
    summary = groq_call(f"Summarize this analysis in simple terms:\n{state['analysis']}")
    return {"summary": summary}

def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back until enough results
    return "summarize"        # Once we have 3+, move to summary

# 4. Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", "summarize": "summarize"})
g.add_edge("summarize", END)

# 5. Run the graph
if __name__ == "__main__":
    app = g.compile()
    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [], "analysis": "",
        "summary": "", "steps_done": 0
    })
    print("\n✅ Final Summary:\n", result["summary"])

🔍 Searching: Quantum Computing
📊 Analyzing 2 results
🔍 Searching: Quantum Computing
📊 Analyzing 4 results
✍️ Generating summary...

✅ Final Summary:
 Here's a summary of the analysis in simple terms:

**What is Quantum Computing?**

* Quantum Computing is a new way of processing information that uses special parts called qubits. Qubits can exist in many states at the same time, making calculations much faster than regular computers.
* Quantum Computing relies on these special connections between qubits to perform calculations that regular computers can't.

**What can Quantum Computing do?**

* It can solve complex problems in many areas like:
	+ Security and encryption
	+ Logistics and finance
	+ Chemical and materials research
	+ Medicine and pharmaceuticals
* It can also do things like predict patterns and recognize images much better than regular computers.

**Challenges in Quantum Computing**

* One big problem is that quantum computers are very unstable and can make mistakes when 

In [11]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.

from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata


# 1. Define State
class SymptomState(TypedDict):
    symptom: str
    observations: list
    analysis: str
    recommendation: str
    steps_done: int


# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):

    groq_api_key = userdata.get("API_KEY")

    if not groq_api_key:
        raise ValueError("Groq API key not found")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        },
    )

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()["choices"][0]["message"]["content"]


# 3. Nodes

# Observation Node
def observe(state: SymptomState):

    print(f"🩺 Observing symptom: {state['symptom']}")

    new_obs = [
        groq_call(f"Give a short health note about {state['symptom']}"),
        groq_call(f"Give another possible cause of {state['symptom']}")
    ]

    return {
        "observations": state["observations"] + new_obs,
        "steps_done": state["steps_done"] + 1
    }


# Analysis Node
def analyze(state: SymptomState):

    print(f"📊 Analyzing {len(state['observations'])} observations")

    joined = "\n".join(state["observations"])

    analysis = groq_call(
        f"Analyze these health notes and extract key insights:\n{joined}"
    )

    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }


# Recommendation Node
def recommend(state: SymptomState):

    print("💡 Generating recommendation")

    rec = groq_call(
        f"Give a simple non-medical recommendation for:\n{state['analysis']}"
    )

    return {
        "recommendation": rec
    }


# Conditional Node
def should_continue(state: SymptomState) -> str:

    if len(state["observations"]) < 3:
        return "observe"

    return "recommend"


# 4. Build Graph
g = StateGraph(SymptomState)

g.add_node("observe", observe)
g.add_node("analyze", analyze)
g.add_node("recommend", recommend)

g.set_entry_point("observe")

g.add_edge("observe", "analyze")

g.add_conditional_edges(
    "analyze",
    should_continue,
    {
        "observe": "observe",
        "recommend": "recommend",
    },
)

g.add_edge("recommend", END)


# 5. Run Graph
if __name__ == "__main__":

    app = g.compile()

    result = app.invoke({
        "symptom": "persistent cough",
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0,
    })

    print("\n✅ Final Recommendation:\n")
    print(result["recommendation"])

🩺 Observing symptom: persistent cough
📊 Analyzing 2 observations
🩺 Observing symptom: persistent cough
📊 Analyzing 4 observations
💡 Generating recommendation

✅ Final Recommendation:

Here's a simple non-medical recommendation for managing a persistent cough:

**Stay hydrated**: Drinking plenty of fluids, such as water, herbal teas, or clear broths, can help thin out mucus and make it easier to cough up. This can provide some temporary relief from coughing and may help to loosen any excess mucus.

**Warm liquids**: Drinking warm liquids, like tea or broth, can help soothe a persistent cough. The steam can also help to moisturize your airways and reduce irritation.

**Rest and relaxation**: Getting enough sleep and managing stress can help your body recover and alleviate a persistent cough. Engage in stress-reducing activities like meditation, reading, or taking short naps.

**Avoid irritants**: Avoid exposure to smoke, dust, and other environmental irritants that can exacerbate a persi

In [1]:
# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output: ✅ Email delivered.

# 👉 Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('API_KEY')
    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def draft_email(state: EmailState):
    print(f"📝 Drafting email for task: {state['task']}")
    draft = groq_call(f"Draft a professional email regarding: {state['task']}")
    return {"draft": draft}

def human_review(state: EmailState):
    # Interrupt node: waits for human approval
    print(f"📧 Draft ready for review:\n\n{state['draft']}\n")
    return {}  # Pauses here until human updates 'approved'

def send_email(state: EmailState):
    if state.get("approved", False):
        print("✅ Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print("❌ Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}

# 4. Build Graph
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]  # Pause before review
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Draft email
app.invoke({"task": "Q3 Report", "draft": "", "approved": False}, thread)

# Step 2: Human reviews draft and resumes
app.invoke({"approved": True}, thread)

📝 Drafting email for task: Q3 Report
📝 Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': "Subject: Q3 Report Update and Performance Review\n\nDear [Manager's Name],\n\nI hope this email finds you well. I am writing to inform you that the Q3 report is now complete and ready for your review. The report provides an overview of our company's performance during the third quarter of [Year] and includes key highlights, financials, and strategic updates.\n\nThe key highlights of the Q3 report include:\n\n- [List key financial metrics, such as revenue growth, profit margins, and cash flow]\n- [Highlight any notable achievements or milestones reached during the quarter]\n- [Outline any challenges or areas for improvement identified during the quarter]\n\nThe Q3 report also includes an update on our current business strategies and objectives, as well as a review of our progress towards key performance indicators (KPIs).\n\nI have attached the complete Q3 report to this email for your review. I would appreciate your feedback and insights on the report a

In [6]:
# ==========================================
# AI REPORT WORKFLOW (HUMAN-IN-THE-LOOP)
# ==========================================

import requests
from google.colab import userdata
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

# 🔑 Load API Key from Colab Secrets
API_KEY = userdata.get('API_KEY')

if not API_KEY:
    raise ValueError("❌ Add 'groq_api_key' in Colab Secrets (🔑 icon in sidebar).")

print("✅ Groq API key loaded.\n")


# ---------------- STATE ---------------- #
class ReportState(TypedDict):
    task: str
    draft: str
    approved: bool


# ---------------- GROQ CALL ---------------- #
def groq_call(prompt: str):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "llama-3.3-70b-versatile",    # ✅ Fixed: updated model
        "messages": [
            {
                "role": "system",
                "content": "You are a professional report writer. Write clear, structured, and detailed reports."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "temperature": 0.7,
        "max_tokens": 800
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"].strip()

    except Exception as e:
        print(f"⚠️  API call failed: {e}")
        return None


# ---------------- NODES ---------------- #

# Node 1: Draft Report
def draft_report(state: ReportState):
    print(f"\n⏳ Generating report for: '{state['task']}' ...")
    print("-" * 65)

    prompt = f"""
Generate a structured professional report on: {state['task']}.

Include the following sections:
1. Introduction
2. Key Points / Findings
3. Analysis
4. Conclusion
5. Recommendations

Make it detailed, clear, and professional.
"""

    draft = groq_call(prompt)

    if not draft:
        draft = (
            f"Report on: {state['task']}\n\n"
            f"1. Introduction\nThis report covers {state['task']}.\n\n"
            f"2. Key Points\n- Point 1\n- Point 2\n- Point 3\n\n"
            f"3. Conclusion\nFurther analysis is recommended.\n\n"
            f"4. Recommendations\nConsult relevant experts for detailed guidance."
        )

    print("\n📄 Draft Report:\n")
    print(draft)
    print("-" * 65)

    return {"draft": draft}


# Node 2: Human Review
def review_report(state: ReportState):
    print("\n🔍 Please review the report above.")

    while True:
        decision = input("\n👉 Approve this report? (yes/no): ").strip().lower()

        if decision in ["yes", "y"]:
            print("✅ You approved the report.")
            return {"approved": True}

        elif decision in ["no", "n"]:
            print("❌ You rejected the report.")
            return {"approved": False}

        else:
            print("⚠️  Invalid input. Please type 'yes' or 'no'.")


# Node 3: Publish or Reject
def publish_report(state: ReportState):
    print("\n" + "=" * 65)

    if state["approved"]:
        print("🚀 Report Approved & Published Successfully!")
        print(f"📬 Topic: {state['task']}")
    else:
        print("🗑️  Report Rejected. Draft discarded.")
        print(f"📌 Topic cancelled: {state['task']}")

    print("=" * 65)
    return {"task": state["task"]}


# ---------------- GRAPH SETUP ---------------- #

builder = StateGraph(ReportState)

builder.add_node("draft",   draft_report)
builder.add_node("review",  review_report)
builder.add_node("publish", publish_report)

builder.set_entry_point("draft")

builder.add_edge("draft",   "review")
builder.add_edge("review",  "publish")
builder.add_edge("publish", END)

memory = MemorySaver()
app = builder.compile(checkpointer=memory)


# ---------------- RUN ---------------- #
def run_report_assistant():
    print("=" * 65)
    print("   📊 AI Report Assistant — Human-in-the-Loop")
    print("=" * 65)

    thread = 0  # thread counter for unique sessions

    while True:
        task = input("\n📌 Enter report topic (or 'exit' to quit): ").strip()

        if task.lower() in ["exit", "quit", "bye"]:
            print("\n👋 Goodbye!")
            break

        if not task:
            print("⚠️  Topic cannot be empty. Please try again.")
            continue

        thread += 1  # new unique thread per report

        state = {
            "task": task,
            "draft": "",
            "approved": False
        }

        # ✅ Fixed: pass thread_id in configurable
        app.invoke(
            state,
            config={"configurable": {"thread_id": str(thread)}}
        )

        print("\n🔄 Ready for next report...\n")


# ---------------- MAIN ---------------- #
if __name__ == "__main__":
    run_report_assistant()

✅ Groq API key loaded.

   📊 AI Report Assistant — Human-in-the-Loop

📌 Enter report topic (or 'exit' to quit): application to manager for salary hike

⏳ Generating report for: 'application to manager for salary hike' ...
-----------------------------------------------------------------

📄 Draft Report:

**Application to Manager for Salary Hike Report**

**Introduction**

As a dedicated and hardworking employee, I am submitting this report to formally request a salary hike. Over the past [X] months/years, I have been an integral part of the team, consistently delivering high-quality work, taking on additional responsibilities, and contributing to the growth and success of the organization. This report aims to provide a comprehensive overview of my achievements, the current market standards, and the justification for a salary increase.

**Key Points / Findings**

The following key points support my application for a salary hike:

1. **Consistent High Performance**: I have consistently r